In [ ]:
%load_ext autoreload
%autoreload 2

# Cardiac Imaging in Medicare — main driver

Thin driver: all logic lives in `src/cardiac_imaging/`. Fix bugs there, not in this notebook.

Run headless (Claude Code does this in the debug loop):

```bash
papermill notebooks/main.ipynb runs/out.ipynb
```

Data: local DuckDB `~/Downloads/cms_parquet/cms.duckdb` (claims view over Parquet + reference tables), opened read-only.

In [ ]:
# Parameters (papermill can override: papermill ... -p DATABASE /path/to/cms.duckdb)
DATABASE = None        # None = config default (~/Downloads/cms_parquet/cms.duckdb or $CARDIAC_DB)
REPORT_ERRORS = True   # write logs/last_error.md on failure (+ GitHub issue if CARDIAC_GITHUB_TOKEN is set)

In [ ]:
# Find the repo, load .env, make src/ importable
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
env_file = ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1)
            if v.strip():
                os.environ.setdefault(k.strip(), v.strip())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

In [ ]:
# Connect read-only and check row counts per year
from cardiac_imaging.db.connection import get_connection, require_tables
con = get_connection(DATABASE)
require_tables(con)
con.sql("SELECT year, COUNT(*) AS rows FROM partb_provider_service GROUP BY year ORDER BY year").show()

In [ ]:
# Run the full analysis → figures in outputs/
from cardiac_imaging import pipeline
ctx = pipeline.run_all(con=con, report_errors=REPORT_ERRORS)

Re-run one step after a fix (reuses earlier results in `ctx`): `dict(pipeline.STEPS)['figures3_7_state_maps'](ctx)`